# Day 3: Deploying for Research

**LLMs for Social Science** | Oxford Spring School

| Day | Theme | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| 2 | From Models to Tools | Done |
| **3** | **Deploying for Research** | **Today** |
| 4 | Social Science Applications | Next |
| 5 | Agentic Workflows | |

## Why this matters

Yesterday you classified political tweets with multiple prompting strategies and discovered that prompt wording alone can shift results by 5-15%. Today you learn how to **validate** those results rigorously, and when prompting is not enough, how to **fine-tune** a model to your task.

You will fine-tune two fundamentally different types of model — an **encoder** (DeBERTa) and a **decoder** (Qwen) — and compare them head-to-head on the same task. By the end of the day, you will know which to reach for and why.

## Today's outline

- **Setup:** Load data and run zero-shot classification (~15 min)
- **Section 1:** Validation — Can You Trust Your Results? (~45 min)
- **Section 2:** Encoder Fine-Tuning with DeBERTa (~35 min)
- **Section 3:** Decoder Fine-Tuning with LoRA (~35 min)
- **Section 4:** The Comparison (~10 min)

## Setup

In [ ]:
#@title Install dependencies
!pip install -q transformers==4.46.3 accelerate datasets peft trl bitsandbytes
!pip install -q torch pandas scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, cohen_kappa_score,
    confusion_matrix, accuracy_score
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title Load dataset and create splits
from sklearn.model_selection import train_test_split

DATA_PATH = 'https://raw.githubusercontent.com/antndlcrx/oss_2024/main/data/WM_tweets_groundtruth.csv'
wm_data = pd.read_csv(DATA_PATH)
wm_data['stance_cat'] = wm_data['stance'].map({1: 'support', 0: 'oppose'})
wm_data['sentiment_cat'] = wm_data['sentiment'].map({1.0: 'positive', 0.0: 'negative'})
wm_data['text_cleaned'] = wm_data['text'].str.replace(r'http\S+|www.\S+', '', case=False, regex=True).str.strip()

# Hold out 250 tweets for zero-shot evaluation (separate from fine-tuning data)
zs_sample = wm_data.sample(n=250, random_state=99)
remaining = wm_data.drop(zs_sample.index)

# Train/val/test splits for fine-tuning (smaller set for faster training)
ft_sample = remaining.sample(n=3000, random_state=42)
train_data, temp_data = train_test_split(ft_sample, test_size=0.2, random_state=42, stratify=ft_sample['stance_cat'])
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['stance_cat'])

print(f"Zero-shot evaluation: {len(zs_sample)} tweets")
print(f"Fine-tuning — Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

## Zero-shot classification

Before we can validate or improve anything, we need baseline results. Let's load the same model from Day 2 and classify 250 tweets with your prompt.

In [ ]:
#@title Load Qwen2.5-1.5B-Instruct
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
zs_tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
zs_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

instruct_pipe = pipeline(
    "text-generation",
    model=zs_model,
    tokenizer=zs_tokenizer,
    device_map="auto",
)
print(f"Loaded {model_name}")

In [ ]:
#@title Classification helpers
def extract_label(response_text, labels):
    """Find the first matching label in the model's response."""
    resp = response_text.lower().strip()
    for label in labels:
        if label in resp:
            return label
    return "unknown"


def classify_batch(pipe, user_messages, max_new_tokens=30):
    """Run batch inference on a list of user messages. Returns raw response strings."""
    prompts = [
        pipe.tokenizer.apply_chat_template(
            [{"role": "user", "content": msg}],
            tokenize=False, add_generation_prompt=True
        )
        for msg in user_messages
    ]
    outputs = pipe(
        prompts,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        batch_size=len(prompts),
    )
    return [out[0]['generated_text'][len(p):].strip() for out, p in zip(outputs, prompts)]


def run_classification(data, prompt_template, labels=("support", "oppose"), batch_size=8):
    """Classify all tweets using the given prompt template.
    Returns a list of predicted labels.
    """
    messages = [prompt_template.format(text=row['text_cleaned']) for _, row in data.iterrows()]
    all_preds = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Classifying"):
        batch = messages[i:i+batch_size]
        responses = classify_batch(instruct_pipe, batch)
        all_preds.extend([extract_label(r, labels) for r in responses])
    return all_preds

print("Classification helpers ready.")

### Your prompt

Paste the prompt you developed yesterday. If you don't have it, use the default below. The `{text}` placeholder will be filled with each tweet.

In [ ]:
# Paste your Day 2 prompt here, or use this default.
PROMPT_TEMPLATE = (
    "Does the author of the following tweet support or oppose the Women's March? "
    "Answer with one word: support or oppose.\n\n"
    "Tweet: {text}"
)

# Sanity check: try it on one tweet
sample_tweet = zs_sample['text_cleaned'].iloc[0]
print(f"Tweet: {sample_tweet[:120]}...")
print(f"\nFormatted prompt:\n{PROMPT_TEMPLATE.format(text=sample_tweet)}")

In [ ]:
#@title Run zero-shot classification
# Run zero-shot classification on 250 tweets
zs_preds = run_classification(zs_sample, PROMPT_TEMPLATE)

zs_results = zs_sample.copy().reset_index(drop=True)
zs_results['model_prediction'] = zs_preds

# Quick summary
zs_acc = accuracy_score(zs_results['stance_cat'], zs_results['model_prediction'])
zs_kappa = cohen_kappa_score(zs_results['stance_cat'], zs_results['model_prediction'])
n_unknown = sum(p == "unknown" for p in zs_preds)

print(f"Zero-shot results on {len(zs_results)} tweets:")
print(f"  Accuracy: {zs_acc:.3f}")
print(f"  Kappa:    {zs_kappa:.3f}")
if n_unknown > 0:
    print(f"  Unknown:  {n_unknown} (model did not return 'support' or 'oppose')")
print()
print(classification_report(zs_results['stance_cat'], zs_results['model_prediction'], digits=3))

Good. You have your baseline. Now: can you trust these numbers?

In [ ]:
#@title Free GPU memory
# Free GPU memory — we don't need the zero-shot model until LoRA
del zs_model, instruct_pipe
torch.cuda.empty_cache()
print("Freed GPU memory for fine-tuning.")

---

# Section 1: Validation
*Can you trust your results?*

---

You just classified 250 tweets. The model gave you numbers. But numbers without validation are just numbers.

Social scientists know this: when you use human coders, you compute inter-coder reliability. LLM-based classification is no different. The model is another coder, and it needs the same scrutiny.

### Exercise 1: Be the Gold Standard

Before looking at any metrics, you need to experience the task yourself. Below are 20 tweets. For each one, decide: does the author **support** or **oppose** the Women's March?

Be honest. Some are genuinely hard. That difficulty is the point.

In [ ]:
# Read each tweet and assign your label: "support" or "oppose"
label_sample = zs_results.head(20).copy()

for i, (_, row) in enumerate(label_sample.iterrows()):
    print(f"[{i:2d}] {row['text_cleaned'][:150]}")
    print()

# Fill in your labels (20 strings: "support" or "oppose")
my_labels = [
    "",  # 0
    "",  # 1
    "",  # 2
    "",  # 3
    "",  # 4
    "",  # 5
    "",  # 6
    "",  # 7
    "",  # 8
    "",  # 9
    "",  # 10
    "",  # 11
    "",  # 12
    "",  # 13
    "",  # 14
    "",  # 15
    "",  # 16
    "",  # 17
    "",  # 18
    "",  # 19
]

In [ ]:
# Now let's see how you did.
filled = [l for l in my_labels if l in ("support", "oppose")]
if len(filled) < 10:
    print(f"Only {len(filled)} labels filled. Please label at least 10 tweets above.")
else:
    n = len(filled)
    gold = label_sample['stance_cat'].values[:n]
    model_preds = label_sample['model_prediction'].values[:n]
    human = filled[:n]

    # You vs gold standard
    human_kappa = cohen_kappa_score(gold, human)
    human_acc = accuracy_score(gold, human)

    # Model vs gold standard (same tweets)
    model_kappa = cohen_kappa_score(gold, model_preds)
    model_acc = accuracy_score(gold, model_preds)

    # You vs model
    you_vs_model = cohen_kappa_score(human, model_preds)

    print(f"On {n} tweets:")
    print(f"  You vs. gold standard:   kappa = {human_kappa:.3f}  (accuracy: {human_acc:.0%})")
    print(f"  Model vs. gold standard: kappa = {model_kappa:.3f}  (accuracy: {model_acc:.0%})")
    print(f"  You vs. model:           kappa = {you_vs_model:.3f}")
    print()
    # Which tweets did YOU disagree on?
    print("Tweets where you disagreed with the gold standard:")
    for i in range(n):
        if human[i] != gold[i]:
            print(f"  [{i}] You: {human[i]}, Gold: {gold[i]}")
            print(f"       {label_sample['text_cleaned'].iloc[i][:120]}")
            print()

Notice what just happened. You, a human who understands sarcasm, context, and political nuance, still disagreed with the gold standard on some tweets. That is your **inter-coder reliability ceiling**. If your kappa with the gold standard is 0.80, expecting the model to exceed 0.80 is unreasonable.

### Why not just use accuracy?

Imagine 90 out of 100 tweets in your dataset are "support." A model that blindly predicts "support" *every single time* gets 90% accuracy. It has learned nothing — but accuracy says it is excellent.

The problem: most of that 90% was **free**. Any model would get it just by following the base rates. Accuracy cannot tell you whether your model is doing anything useful beyond that.

### Cohen's kappa: only count the hard agreement

Kappa asks a simple question: **of the improvement that was possible beyond just guessing, how much did you actually achieve?**

Think of it this way. If random guessing gives you 80% agreement (the floor), and perfect agreement is 100% (the ceiling), there are 20 percentage points of "hard" agreement available. If your model gets 90%, it captured half of that hard part. Kappa = 0.5.

A concrete example: suppose your always-"support" model scores 90% accuracy. The floor (expected agreement from the base rates alone) is also ~90%. So kappa = (0.90 − 0.90) / (1 − 0.90) = **0.0**. Kappa correctly tells you: this model adds nothing.

Now a model that actually reads the tweets and gets 95% accuracy on the same data: kappa = (0.95 − 0.90) / (1 − 0.90) = **0.5**. It captured half the hard cases.

The formula is just notation for this:

$$\kappa = \frac{p_o - p_e}{1 - p_e} = \frac{\text{observed agreement} - \text{floor}}{\text{ceiling} - \text{floor}}$$

Common benchmarks: $\kappa > 0.8$ is "almost perfect," 0.6–0.8 is "substantial," 0.4–0.6 is "moderate."

### Exercise 2: Predict the Errors

Now let's look at where the *model* went wrong. But instead of just reading errors passively, try to **predict the error pattern** before reading each tweet.

Common LLM error patterns:
- **A. Sarcasm/irony:** positive words, negative intent (or vice versa)
- **B. Mixed signals:** both supportive and opposing language in the same tweet
- **C. Too short/ambiguous:** not enough context to decide
- **D. Indirect stance:** author reports or quotes another position
- **E. Other/unclear**

For each misclassified tweet below, write the letter of the error pattern you think applies.

In [ ]:
misclassified = zs_results[
    zs_results['stance_cat'] != zs_results['model_prediction']
].head(10).copy()

print(f"Showing 10 of {len(zs_results[zs_results['stance_cat'] != zs_results['model_prediction']])} misclassified tweets:\n")
for i, (_, row) in enumerate(misclassified.iterrows()):
    print(f"[{i}] True: {row['stance_cat']} | Model: {row['model_prediction']}")
    print(f"    {row['text_cleaned'][:150]}")
    print()

# YOUR ERROR PATTERN LABELS (A, B, C, D, or E for each tweet)
error_patterns = [
    "",  # tweet 0
    "",  # tweet 1
    "",  # tweet 2
    "",  # tweet 3
    "",  # tweet 4
    "",  # tweet 5
    "",  # tweet 6
    "",  # tweet 7
    "",  # tweet 8
    "",  # tweet 9
]

In [ ]:
# What does the distribution of error patterns tell you?
from collections import Counter
if any(error_patterns):
    counts = Counter(p.upper() for p in error_patterns if p)
    pattern_names = {"A": "Sarcasm/irony", "B": "Mixed signals",
                     "C": "Too short", "D": "Indirect stance", "E": "Other"}
    print("Your error pattern breakdown:")
    for letter, count in counts.most_common():
        name = pattern_names.get(letter, "?")
        print(f"  {letter}. {name}: {count}")
    print()
    print("Which pattern is most common? This tells you what to target:")
    print("  Sarcasm/irony  -> chain-of-thought prompting or more examples")
    print("  Mixed signals  -> clearer instructions about what to prioritize")
    print("  Too short      -> these may be genuinely ambiguous (accept the loss)")
    print("  Indirect stance -> instruction to classify the author stance, not reported stance")
else:
    print("Fill in error_patterns above to see the breakdown.")

# Also: is the model biased toward one direction?
cm = confusion_matrix(zs_results['stance_cat'], zs_results['model_prediction'], labels=['support', 'oppose'])
print(f"\nConfusion matrix (all {len(zs_results)} tweets):")
print(f"              Predicted support  Predicted oppose")
print(f"True support       {cm[0,0]:5d}            {cm[0,1]:5d}")
print(f"True oppose        {cm[1,0]:5d}            {cm[1,1]:5d}")

### Exercise 3: Design Your Validation

You now have hands-on experience with what validation involves. For your own research, you would need a plan. Answer these questions for a **hypothetical project** where you want to classify 10,000 newspaper articles as "pro-government" or "anti-government."

1. How many articles would you label by hand for your gold standard? (Think: enough to get stable kappa estimates.)
2. How would you sample them? (Random? Stratified by source? Over-sample hard cases?)
3. What kappa threshold would you set as "good enough" before proceeding?
4. Based on what you saw in Exercise 2, what error patterns would you specifically check for?

In [ ]:
# Write your validation plan.
# This is not code, just structured thinking.

validation_plan = {
    "gold_standard_size": "",      # e.g., "200 articles"
    "sampling_strategy": "",       # e.g., "stratified by newspaper source"
    "kappa_threshold": "",         # e.g., "0.7 (substantial agreement)"
    "error_patterns_to_check": "", # e.g., "sarcasm in opinion columns, ..."
}

for k, v in validation_plan.items():
    print(f"  {k}: {v or '[fill in]'}")

### Systematic biases to watch for

Beyond the random errors you just categorized, LLMs exhibit *systematic* biases:

**Positional bias:** When given options ("support or oppose"), models tend to favor the option listed first. Your Day 2 sensitivity exercise may have shown this.

**Sycophancy bias:** Models tend toward the more "positive" or "agreeable" option. In stance detection, this means over-predicting support.

**Cultural bias:** Models trained on English-language Western data may misinterpret rhetorical conventions from other contexts.

These are threats to measurement validity. Document them, and where possible, correct them (e.g., by rotating label order in prompts).

**Section takeaway:** Treat the LLM as another coder, not as ground truth. Compute kappa (not just accuracy). Analyze errors qualitatively. Know your ceiling by measuring human-human agreement. This applies whether you use prompting, fine-tuning, or any other approach.

# Section 2: The Fine-Tuning Decision

## When to prompt vs. RAG vs. fine-tune

You now have concrete evidence of how well prompting works for your task and where it fails. What do you do next?

**Prompting** is the right starting point. Stick with it when zero-shot or few-shot performance meets your threshold, when you need flexibility, or when you have little labeled data.

**RAG** (Day 4) is the right choice when the model needs domain knowledge: your corpus of parliamentary records, legal texts, or any collection too large for the context window.

**Fine-tuning** is the right choice when prompting hits a ceiling and your validation (Section 1) shows systematic errors, when you need consistent output formatting, when you have labeled data (hundreds to thousands), or when domain-specific language is not handled well by prompting.

Your Section 1 results should help you decide: if kappa is already 0.8+ with prompting, fine-tuning may not be worth the effort. If it is 0.6, there is room to improve.

## Two kinds of fine-tuning

But "fine-tuning" is not one thing. There are two fundamentally different approaches, and choosing the wrong one wastes time and compute.

**Encoder models** (BERT, DeBERTa, RoBERTa) read text **bidirectionally** — every token attends to every other token simultaneously. You add a classification head on top and train the whole thing to output a label. These models are small (44M–400M parameters), fast to train (2 minutes on a free Colab GPU), and often the most accurate option for classification.

**Decoder models** (Qwen, Llama, GPT) read text **left-to-right** and generate tokens one at a time. You fine-tune them by teaching them to produce the right answer in conversation format (LoRA/QLoRA). These models are large (1B–70B+ parameters), slower to train, but flexible — the same fine-tuned model can classify, extract, and summarize.

We will do both, on the same task, and compare them. Encoders first — they are faster, so you will get results quickly.

## Encoder fine-tuning with DeBERTa

DeBERTa-v3-small has 44 million parameters. Compare that to the 1.5 billion in Qwen2.5 — more than 30 times smaller. Yet for classification specifically, it is often competitive or better, because the entire architecture is designed for understanding text, not generating it.

The training data format is simple: each example is a text string and an integer label. No conversation formatting, no prompt engineering. The model learns to map text directly to categories.

### Exercise 4: Prepare the Training Data

Encoder fine-tuning needs a different data format than what you used for prompting. Instead of a conversation with instructions, the model simply receives raw text and learns to map it to a label index.

Write a function that converts the dataframe into the format DeBERTa expects: a `text` field and an integer `label` field (0 for oppose, 1 for support).

In [ ]:
label2id = {'oppose': 0, 'support': 1}
id2label = {0: 'oppose', 1: 'support'}

def prepare_encoder_data(df, label2id):
    """
    Convert a dataframe with 'text_cleaned' and 'stance_cat' columns
    into a dict with 'text' (list of strings) and 'label' (list of ints).

    This is all an encoder needs. No instructions, no conversation format.
    """
    # YOUR CODE HERE
    pass

# Test it
sample_output = prepare_encoder_data(train_data.head(3), label2id)
for i in range(3):
    print(f"  text: {sample_output['text'][i][:80]}...")
    print(f"  label: {sample_output['label'][i]} ({id2label[sample_output['label'][i]]})")
    print()

In [ ]:
#@title Tokenize for DeBERTa
from datasets import Dataset
from transformers import AutoTokenizer

# Build HuggingFace datasets
train_enc = Dataset.from_dict(prepare_encoder_data(train_data, label2id))
val_enc = Dataset.from_dict(prepare_encoder_data(val_data, label2id))
test_enc = Dataset.from_dict(prepare_encoder_data(test_data, label2id))

# Tokenize
deberta_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)

def tokenize_fn(examples):
    return deberta_tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_enc = train_enc.map(tokenize_fn, batched=True)
val_enc = val_enc.map(tokenize_fn, batched=True)
test_enc = test_enc.map(tokenize_fn, batched=True)
print(f"Tokenized: Train={len(train_enc)}, Val={len(val_enc)}, Test={len(test_enc)}")

### Exercise 5: Train, Evaluate, and Probe

Training DeBERTa-v3-small takes about 2 minutes on a T4 GPU. While it runs, compare what is happening here to what you did earlier:

- Earlier, you *described* the task in natural language and the model *interpreted* your instructions.
- Here, you show the model raw text and correct labels, and it *learns the mapping directly*.

No prompt sensitivity. No instruction ambiguity. The tradeoff: this model can only do this one task.

You will see a load report with UNEXPECTED and MISSING keys. This is normal: the model is swapping its word-prediction head (from pre-training) for a classification head (your task). The shared body of the model — the part that understands language — transfers over intact.

#### Reading the training config

The code below sets several **hyperparameters** — settings that control how training works. If you have not trained a model before, here is what each one does and why it matters:

**`num_train_epochs=3`** — How many times the model sees the entire training set. One pass through all the data is one epoch. Too few and the model underfits (has not learned enough); too many and it overfits (memorizes the training data instead of learning general patterns). For fine-tuning a pretrained model, 2–4 epochs is typical — the model already understands language, it just needs to learn your task.

**`per_device_train_batch_size=16`** — How many examples the model processes before updating its weights. Larger batches give more stable updates but use more GPU memory. 16 is a safe default for a small model on a T4 GPU.

**`learning_rate=2e-5`** — How big a step the model takes when updating its weights after each batch. This is the single most important hyperparameter. Too high and the model overshoots and destroys what it already knows; too low and it barely learns. For fine-tuning pretrained transformer models, the range 1e-5 to 5e-5 is well-established — much smaller than training from scratch, because you are making small adjustments to an already capable model.

**`eval_strategy="epoch"`** — Check performance on the validation set after each epoch, so you can see if the model is improving or starting to overfit.

**`load_best_model_at_end=True`** — After all epochs, go back and keep the version that scored best on validation, not necessarily the last one. This is your insurance against overfitting in later epochs.

**`fp16=False`** — Half-precision (fp16) usually speeds up training with negligible quality loss, and we use it for LoRA later. But DeBERTa-v3 specifically is numerically unstable with fp16 — its disentangled attention mechanism produces values that overflow in half-precision, leading to NaN loss and a broken model. So we train in full precision (fp32) here. It is a bit slower but works correctly. This is a known quirk of DeBERTa-v3, not a general rule.

In [ ]:
#@title Train DeBERTa
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name, num_labels=2, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1}

training_args = TrainingArguments(
    output_dir="./deberta_stance",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    fp16=False,  # DeBERTa-v3 is unstable with fp16
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=val_enc,
    compute_metrics=compute_metrics,
)

print("Training DeBERTa-v3-small...")
deberta_trainer.train()
print("Training complete.")

In [ ]:
#@title Evaluate DeBERTa
# Evaluate on test set
test_output = deberta_trainer.predict(test_enc)
deberta_preds = np.argmax(test_output.predictions, axis=-1)
deberta_labels = [id2label[p] for p in deberta_preds]
deberta_true = test_data['stance_cat'].values

deberta_accuracy = accuracy_score(deberta_true, deberta_labels)
deberta_kappa = cohen_kappa_score(deberta_true, deberta_labels)

print("DeBERTa-v3-small test results:")
print(classification_report(deberta_true, deberta_labels, digits=3))
print(f"Cohen's kappa: {deberta_kappa:.3f}")

#### The probe: does DeBERTa fix the hard cases?

Remember the tweets the zero-shot model got wrong? Let's see if a dedicated classification model handles them better.

In [ ]:
# Run DeBERTa on zero-shot errors
wrong_zs = zs_results[
    zs_results['stance_cat'] != zs_results['model_prediction']
].copy()

if len(wrong_zs) > 0:
    probe_texts = wrong_zs['text_cleaned'].head(20).tolist()
    probe_true = wrong_zs['stance_cat'].head(20).tolist()
    probe_zs = wrong_zs['model_prediction'].head(20).tolist()

    # Tokenize and predict
    probe_enc = Dataset.from_dict({'text': probe_texts, 'label': [0]*len(probe_texts)})
    probe_enc = probe_enc.map(tokenize_fn, batched=True)
    probe_output = deberta_trainer.predict(probe_enc)
    probe_deberta = [id2label[p] for p in np.argmax(probe_output.predictions, axis=-1)]

    correct_deberta = sum(t == p for t, p in zip(probe_true, probe_deberta))

    print(f"On {len(probe_texts)} tweets that zero-shot prompting got WRONG:")
    print(f"  Zero-shot:  0/{len(probe_texts)} correct (all wrong by definition)")
    print(f"  DeBERTa:    {correct_deberta}/{len(probe_texts)} correct")
    print()

    # Show examples where DeBERTa fixed the error
    print("Examples where DeBERTa fixed the error:")
    shown = 0
    for i in range(len(probe_texts)):
        if probe_deberta[i] == probe_true[i] and shown < 3:
            print(f"  True: {probe_true[i]} | Zero-shot: {probe_zs[i]} | DeBERTa: {probe_deberta[i]}")
            print(f"  {probe_texts[i][:140]}")
            print()
            shown += 1

    # And examples where it still fails
    still_wrong = [(i, probe_deberta[i]) for i in range(len(probe_texts)) if probe_deberta[i] != probe_true[i]]
    if still_wrong:
        print("Examples where DeBERTa still fails:")
        for i, pred in still_wrong[:2]:
            print(f"  True: {probe_true[i]} | DeBERTa: {pred}")
            print(f"  {probe_texts[i][:140]}")
            print()

Note the training time. DeBERTa trained in about 2 minutes and produced a dedicated classifier. Keep this number in mind — we are about to try the decoder approach, which takes considerably longer.

---

# Section 3: Decoder Fine-Tuning with LoRA

## When you need a decoder

DeBERTa is excellent for classification, but it can *only* classify. If your research requires any of these, you need a decoder:

- **Extracting** structured information (names, dates, events) from text
- **Summarizing** documents or generating explanations alongside labels
- **Multi-task** workflows where the same model classifies, extracts, and explains
- **Open-ended labeling** where categories emerge from the data

The challenge: decoder models are much larger. Qwen2.5-1.5B has 1.5 billion parameters. Fine-tuning all of them would require substantial GPU memory and risk catastrophic forgetting.

**LoRA (Low-Rank Adaptation)** solves this by freezing the entire base model and inserting small trainable matrices ("adapters") into the attention layers. Instead of updating 1.5 billion parameters, you train a few million. This dramatically cuts memory and preserves the model's general capabilities.

**QLoRA** goes further: it loads the base model in 4-bit precision, so the whole thing fits on a free Colab T4.

The connection to prompting is direct: the training data for LoRA is prompt-response pairs, exactly the format you used for zero-shot classification earlier. The difference is that instead of the model seeing your examples at inference time (few-shot), it *learns* from them during training.

### Exercise 6: Format the Training Data

LoRA fine-tuning needs data formatted as **conversations**: a user instruction and the correct assistant response. This is exactly the prompting format from earlier, but with the right answer filled in.

Compare this to Exercise 4. DeBERTa needed raw text and an integer. Here, you need to construct the full instruction — the same one you would type into a chat interface.

Write a function that converts each labeled tweet into a training example.

In [ ]:
def format_for_sft(row):
    """
    Convert a labeled tweet into a chat-format training example.

    Return a list of two message dicts:
      1. A "user" message with the classification instruction + tweet
      2. An "assistant" message with the correct label

    Use the same instruction style from your prompt.
    """
    messages = []  # YOUR CODE HERE
    return messages

# Test on one example
sample = train_data.iloc[0]
for msg in format_for_sft(sample):
    print(f"  [{msg['role']}]: {msg['content'][:120]}")

In [ ]:
#@title Format all training data
train_conversations = [format_for_sft(row) for _, row in train_data.iterrows()]
val_conversations = [format_for_sft(row) for _, row in val_data.iterrows()]
print(f"Formatted {len(train_conversations)} training + {len(val_conversations)} validation examples")
print(f"\nExample conversation:")
for msg in train_conversations[0]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}")

### Exercise 7: Train, Evaluate, and Compare

Training LoRA on Qwen takes about 10–15 minutes on a T4 — considerably longer than DeBERTa, despite training fewer parameters proportionally. This is because the base model is much larger and generates tokens sequentially.

While training runs, think about this: the model is learning from the same instruction-response format you used for zero-shot classification. The difference is that instead of providing examples at inference time (few-shot), the model *internalizes* the patterns during training.

#### New hyperparameters for LoRA

You already know `learning_rate`, `num_train_epochs`, and `fp16` from DeBERTa. The LoRA setup introduces a few new ones:

**`r=16`** — The **rank** of the LoRA adapters. This controls how expressive the adapters are. Higher rank = more trainable parameters = more capacity to learn, but also more memory and risk of overfitting. For most tasks, r=8 to r=32 works well. Think of it as the "width" of the small matrices you are inserting into the frozen model.

**`lora_alpha=32`** — A scaling factor for the adapter outputs. The ratio `lora_alpha / r` determines how much influence the adapters have relative to the frozen base model. A common rule of thumb is to set `lora_alpha = 2 * r`.

**`per_device_train_batch_size=4`** with **`gradient_accumulation_steps=4`** — The model is too large to fit 16 examples in GPU memory at once (unlike DeBERTa). So we process 4 at a time, accumulate the gradients across 4 steps, and then update — giving an *effective* batch size of 16. Same result, just spread over more steps to fit in memory.

**`load_in_4bit=True`** — This is the "Q" in QLoRA. The frozen base model is compressed to 4-bit precision, cutting memory by ~4×. The LoRA adapters themselves stay in full precision. This is why a 1.5B-parameter model fits on a free Colab T4.

In [ ]:
#@title Load Qwen2.5-1.5B-Instruct with QLoRA (4-bit)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
lora_tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
lora_model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto",
)
lora_model = prepare_model_for_kbit_training(lora_model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
lora_model = get_peft_model(lora_model, lora_config)

trainable, total = lora_model.get_nb_trainable_parameters()
print(f"Total parameters:     {total:>12,}")
print(f"Trainable (LoRA):     {trainable:>12,} ({100*trainable/total:.2f}%)")
print(f"GPU memory:           {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
#@title Train LoRA on Qwen
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def conversations_to_dataset(conversations):
    return Dataset.from_dict({"messages": conversations})

train_sft_ds = conversations_to_dataset(train_conversations)
val_sft_ds = conversations_to_dataset(val_conversations)

sft_config = SFTConfig(
    output_dir="./qwen_lora_stance",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size = 16
    learning_rate=2e-4,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    max_seq_length=256,
    report_to="none",
)

trainer = SFTTrainer(
    model=lora_model,
    args=sft_config,
    train_dataset=train_sft_ds,
    eval_dataset=val_sft_ds,
    processing_class=lora_tokenizer,
)

print("Starting LoRA training...")
trainer.train()
print("Training complete.")

In [ ]:
#@title Evaluate LoRA model on test set
lora_model.eval()

def classify_with_lora(texts, model, tokenizer, batch_size=8):
    """Classify texts using the LoRA-adapted model."""
    predictions = []
    for i in tqdm(range(0, len(texts), batch_size), desc="LoRA inference"):
        batch_texts = texts[i:i+batch_size]
        prompts = []
        for text in batch_texts:
            messages = [{"role": "user", "content": (
                "Does the author of the following tweet support or oppose the Women's March? "
                "Answer with one word: support or oppose.\n\n"
                f"Tweet: {text}"
            )}]
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        for j in range(len(batch_texts)):
            prompt_len = inputs['input_ids'][j].ne(tokenizer.pad_token_id).sum()
            resp = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True).lower().strip()
            if "support" in resp: predictions.append("support")
            elif "oppose" in resp: predictions.append("oppose")
            else: predictions.append("unknown")
    return predictions

lora_preds = classify_with_lora(test_data['text_cleaned'].tolist(), lora_model, lora_tokenizer)
lora_accuracy = accuracy_score(test_data['stance_cat'], lora_preds)
lora_kappa = cohen_kappa_score(test_data['stance_cat'], lora_preds)

print("LoRA (Qwen2.5-1.5B) test results:")
print(classification_report(test_data['stance_cat'], lora_preds, digits=3))
print(f"Cohen's kappa: {lora_kappa:.3f}")

#### The probe: did LoRA fix the hard cases?

Same exercise as with DeBERTa — run the fine-tuned decoder on the exact tweets that zero-shot prompting got wrong.

In [ ]:
if len(wrong_zs) > 0:
    probe_lora = classify_with_lora(probe_texts, lora_model, lora_tokenizer)

    correct_lora = sum(t == p for t, p in zip(probe_true, probe_lora))

    print(f"On {len(probe_texts)} tweets that zero-shot prompting got WRONG:")
    print(f"  Zero-shot:   0/{len(probe_texts)} correct (all wrong by definition)")
    print(f"  DeBERTa:     {correct_deberta}/{len(probe_texts)} correct")
    print(f"  LoRA (Qwen): {correct_lora}/{len(probe_texts)} correct")
    print()

    # Show where they disagree
    print("Tweets where DeBERTa and LoRA disagree:")
    shown = 0
    for i in range(len(probe_texts)):
        if probe_deberta[i] != probe_lora[i] and shown < 5:
            print(f"  True: {probe_true[i]} | DeBERTa: {probe_deberta[i]} | LoRA: {probe_lora[i]}")
            print(f"  {probe_texts[i][:140]}")
            print()
            shown += 1

---

# Section 4: The Comparison

You have now experienced three approaches to the same task: zero-shot prompting, encoder fine-tuning (DeBERTa), and decoder fine-tuning (LoRA on Qwen). Let's put them side by side.

In [ ]:
print("=" * 95)
print(f"{'Approach':<30} {'Accuracy':>10} {'Kappa':>8} {'Parameters':>16} {'Train time':>12} {'Flexibility':>12}")
print("-" * 95)
print(f"{'Zero-shot prompting':<30} {zs_acc:>10.3f} {zs_kappa:>8.3f} {'1.5B (none trained)':>16} {'None':>12} {'High':>12}")
print(f"{'DeBERTa-v3-small':<30} {deberta_accuracy:>10.3f} {deberta_kappa:>8.3f} {'44M':>16} {'~2 min':>12} {'Low':>12}")
print(f"{'LoRA (Qwen2.5-1.5B)':<30} {lora_accuracy:>10.3f} {lora_kappa:>8.3f} {'1.5B (few M trained)':>16} {'~10-15 min':>12} {'High':>12}")
print("=" * 95)

### A decision tree

**Start with prompting.** If kappa ≥ your threshold → done.

**If prompting is not enough, ask: is my task pure classification?**
- **Yes** (fixed categories, labeled data available) → **encoder fine-tuning** (DeBERTa). Faster to train, cheaper to run, often more accurate for classification specifically.
- **No** (need extraction, summarization, multi-task, or flexible output) → **decoder fine-tuning** (LoRA). More expensive, but can do everything.

**If you have no labeled data at all** → stay with prompting, or invest in annotation first. Fine-tuning without good data makes things worse, not better.

---

## What you learned today

1. **Validation is not optional.** You experienced it firsthand: you disagreed with the gold standard on some tweets, and that sets a ceiling. Use Cohen's kappa, not just accuracy. Analyze errors qualitatively.

2. **Error patterns are actionable.** Sarcasm, mixed signals, and indirect stance are not just descriptions; each points to a specific improvement strategy.

3. **Encoders are the workhorse for classification.** DeBERTa trained in 2 minutes, uses a fraction of the compute, and is purpose-built for the task. For fixed-category classification with labeled data, start here.

4. **Decoders are the flexible option.** LoRA on Qwen preserves the model's ability to generate, extract, and follow instructions. Use it when your task goes beyond pure classification.

5. **The comparison is the point.** You saw both approaches on the same data, probed the same hard cases, and now have firsthand evidence for when to use which.

## Bridge to Day 4

You now have a complete pipeline: prompt, validate, fine-tune. Day 4 goes beyond classification:

- **Information extraction and summarization:** pulling structured data from unstructured text
- **RAG:** when your corpus exceeds the context window, embeddings (from Day 1) power retrieval
- **LLMs as simulated agents:** can language models stand in for human survey respondents?

---

Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)

---
# Solutions

---

In [ ]:
#@title Solution: Exercise 4 (Prepare Encoder Data)
def prepare_encoder_data(df, label2id):
    return {
        'text': df['text_cleaned'].tolist(),
        'label': [label2id[s] for s in df['stance_cat']],
    }

sample_output = prepare_encoder_data(train_data.head(3), label2id)
for i in range(3):
    print(f"  text: {sample_output['text'][i][:80]}...")
    print(f"  label: {sample_output['label'][i]} ({id2label[sample_output['label'][i]]})")
    print()

In [ ]:
#@title Solution: Exercise 6 (Format SFT Data)
def format_for_sft(row):
    """Convert a labeled tweet into a chat-format training example."""
    return [
        {"role": "user", "content": (
            "Does the author of the following tweet support or oppose the Women's March? "
            "Answer with one word: support or oppose.\n\n"
            f"Tweet: {row['text_cleaned']}"
        )},
        {"role": "assistant", "content": row['stance_cat']}
    ]

sample = train_data.iloc[0]
for msg in format_for_sft(sample):
    print(f"  [{msg['role']}]: {msg['content'][:120]}")

---
*This notebook is part of the [LLMs for Social Science](https://llmsforsocialscience.net) course.*